# RNN Sentiment Classification on IMDb
## Comprehensive Experimental Analysis: Optimizers, Training Epochs, and Model Architectures

**Dataset:** IMDb Movie Reviews (50,000 samples)  
**Framework:** PyTorch  
**Task:** Binary Sentiment Classification (Positive / Negative)

This notebook systematically evaluates:
- **Warm-up** — Baseline Vanilla RNN trained with SGD for 5 epochs  
- **Experiment 1** — Optimizer comparison: SGD vs. Adam vs. Adagrad  
- **Experiment 2** — Effect of training epochs: 5 / 10 / 20 / 50  
- **Experiment 3** — Architecture comparison: FFN-1/2/3, CNN, LSTM, Bi-LSTM  


## 1. Setup & Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing import sequence
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
print(f"GPU available   : {torch.cuda.is_available()}")


In [ ]:
class Config:
    """Global hyperparameters shared across all experiments."""
    VOCAB_SIZE    = 10_000   # top-10k most frequent tokens
    MAX_LENGTH    = 500      # pad / truncate to 500 tokens
    EMBEDDING_DIM = 128      # randomly initialised, trained jointly
    BATCH_SIZE    = 32
    HIDDEN_DIM    = 256      # RNN / LSTM / FFN hidden size
    DROPOUT       = 0.3
    LEARNING_RATE = 0.001
    SEED          = 42

config = Config()
print("Hyperparameter Configuration:")
for k, v in vars(Config).items():
    if not k.startswith('_'):
        print(f"  {k:<20} = {v}")


## 2. Data Loading & Preprocessing

In [ ]:
print("Loading IMDb dataset...")

try:
    from tensorflow.keras.datasets import imdb
    (X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=config.VOCAB_SIZE)
    print("✓ IMDb dataset loaded successfully")
except Exception:
    print("⚠ Network unavailable — generating synthetic data for testing")
    X_train = np.random.randint(1, config.VOCAB_SIZE, (25000, config.MAX_LENGTH))
    y_train = np.random.randint(0, 2, 25000)
    X_test  = np.random.randint(1, config.VOCAB_SIZE, (25000, config.MAX_LENGTH))
    y_test  = np.random.randint(0, 2, 25000)

# Pad / truncate sequences
X_train_padded = sequence.pad_sequences(X_train, maxlen=config.MAX_LENGTH, padding='post')
X_test_padded  = sequence.pad_sequences(X_test,  maxlen=config.MAX_LENGTH, padding='post')

y_train = np.array(y_train, dtype=np.float32)
y_test  = np.array(y_test,  dtype=np.float32)

# 70 / 30 train-validation split
split_idx      = int(len(X_train_padded) * 0.7)
X_train_split  = X_train_padded[:split_idx]
y_train_split  = y_train[:split_idx]
X_valid        = X_train_padded[split_idx:]
y_valid        = y_train[split_idx:]

# PyTorch tensors
X_train_tensor = torch.LongTensor(X_train_split)
y_train_tensor = torch.FloatTensor(y_train_split).unsqueeze(1)
X_valid_tensor = torch.LongTensor(X_valid)
y_valid_tensor = torch.FloatTensor(y_valid).unsqueeze(1)
X_test_tensor  = torch.LongTensor(X_test_padded)
y_test_tensor  = torch.FloatTensor(y_test).unsqueeze(1)

# DataLoaders
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor),
                          batch_size=config.BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(TensorDataset(X_valid_tensor, y_valid_tensor),
                          batch_size=config.BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test_tensor,  y_test_tensor),
                          batch_size=config.BATCH_SIZE, shuffle=False)

print(f"\nDataset split summary:")
print(f"  Training   samples : {len(X_train_split):,}")
print(f"  Validation samples : {len(X_valid):,}")
print(f"  Test       samples : {len(X_test_padded):,}")
print(f"  Total              : {len(X_train_split)+len(X_valid)+len(X_test_padded):,}")


## 3. Model Architectures

Six architectures are implemented and compared:

| # | Model   | Description |
|---|---------|-------------|
| 1 | **BaselineRNN** | Vanilla RNN (warm-up baseline) |
| 2 | **FFN-1** | 1-hidden-layer FFN (500 units), flattened embeddings |
| 3 | **FFN-2** | 2-hidden-layer FFN (500 → 300) |
| 4 | **FFN-3** | 3-hidden-layer FFN (500 → 300 → 200) |
| 5 | **CNN**   | Three parallel 1-D convolutions, kernel sizes 1 / 2 / 3 |
| 6 | **LSTM**  | Unidirectional LSTM |
| 7 | **Bi-LSTM** | Bidirectional LSTM |


In [ ]:
# ── Baseline RNN ─────────────────────────────────────────────────────────────
class BaselineRNN(nn.Module):
    """Vanilla RNN — warm-up baseline model."""
    def __init__(self, vocab_size, emb_dim, hidden_dim, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.rnn       = nn.RNN(emb_dim, hidden_dim, batch_first=True)
        self.fc        = nn.Linear(hidden_dim, 1)
        self.dropout   = nn.Dropout(dropout)
        self.sigmoid   = nn.Sigmoid()

    def forward(self, text):
        emb = self.dropout(self.embedding(text))
        _, hidden = self.rnn(emb)
        return self.sigmoid(self.fc(hidden.squeeze(0)))


# ── Feed-Forward Networks ────────────────────────────────────────────────────
class FFN1(nn.Module):
    """1-hidden-layer FFN: flatten -> 500 -> 1."""
    def __init__(self, vocab_size, emb_dim, hidden_dim, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.fc1     = nn.Linear(emb_dim * 500, hidden_dim)
        self.fc2     = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)
        self.relu    = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, text):
        emb = self.embedding(text).view(text.size(0), -1)
        h   = self.relu(self.fc1(self.dropout(emb)))
        return self.sigmoid(self.fc2(self.dropout(h)))


class FFN2(nn.Module):
    """2-hidden-layer FFN: flatten -> 500 -> 300 -> 1."""
    def __init__(self, vocab_size, emb_dim, hidden_dim, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.fc1     = nn.Linear(emb_dim * 500, hidden_dim)
        self.fc2     = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3     = nn.Linear(hidden_dim // 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.relu    = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, text):
        emb = self.embedding(text).view(text.size(0), -1)
        h   = self.relu(self.fc1(self.dropout(emb)))
        h   = self.relu(self.fc2(self.dropout(h)))
        return self.sigmoid(self.fc3(self.dropout(h)))


class FFN3(nn.Module):
    """3-hidden-layer FFN: flatten -> 500 -> 300 -> 200 -> 1."""
    def __init__(self, vocab_size, emb_dim, hidden_dim, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.fc1     = nn.Linear(emb_dim * 500, hidden_dim)
        self.fc2     = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3     = nn.Linear(hidden_dim // 2, hidden_dim // 4)
        self.fc4     = nn.Linear(hidden_dim // 4, 1)
        self.dropout = nn.Dropout(dropout)
        self.relu    = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, text):
        emb = self.embedding(text).view(text.size(0), -1)
        h   = self.relu(self.fc1(self.dropout(emb)))
        h   = self.relu(self.fc2(self.dropout(h)))
        h   = self.relu(self.fc3(self.dropout(h)))
        return self.sigmoid(self.fc4(self.dropout(h)))


# ── CNN ──────────────────────────────────────────────────────────────────────
class CNN(nn.Module):
    """1-D CNN with three parallel filters (kernel sizes 1, 2, 3) + global max-pool."""
    def __init__(self, vocab_size, emb_dim, num_filters=100, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.convs     = nn.ModuleList([
            nn.Conv1d(emb_dim, num_filters, kernel_size=k) for k in [1, 2, 3]
        ])
        self.fc      = nn.Linear(num_filters * 3, 1)
        self.dropout = nn.Dropout(dropout)
        self.relu    = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, text):
        emb = self.embedding(text).transpose(1, 2)          # (B, emb_dim, L)
        pooled = [self.relu(conv(emb)).max(dim=2)[0] for conv in self.convs]
        h = torch.cat(pooled, dim=1)
        return self.sigmoid(self.fc(self.dropout(h)))


# ── LSTM ─────────────────────────────────────────────────────────────────────
class LSTMModel(nn.Module):
    """Unidirectional LSTM."""
    def __init__(self, vocab_size, emb_dim, hidden_dim, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm      = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.fc        = nn.Linear(hidden_dim, 1)
        self.dropout   = nn.Dropout(dropout)
        self.sigmoid   = nn.Sigmoid()

    def forward(self, text):
        emb = self.dropout(self.embedding(text))
        _, (hidden, _) = self.lstm(emb)
        return self.sigmoid(self.fc(hidden.squeeze(0)))


# ── Bi-LSTM ───────────────────────────────────────────────────────────────────
class BiLSTM(nn.Module):
    """Bidirectional LSTM (concatenates forward & backward final hidden states)."""
    def __init__(self, vocab_size, emb_dim, hidden_dim, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.bilstm    = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc        = nn.Linear(hidden_dim * 2, 1)
        self.dropout   = nn.Dropout(dropout)
        self.sigmoid   = nn.Sigmoid()

    def forward(self, text):
        emb = self.dropout(self.embedding(text))
        _, (hidden, _) = self.bilstm(emb)
        hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.sigmoid(self.fc(hidden))


print("✓ All model architectures defined successfully")


## 4. Training & Evaluation Utilities

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Run one training epoch with gradient clipping (max_norm=1.0)."""
    model.train()
    total_loss, n_batches = 0.0, 0
    for batch_text, batch_label in loader:
        batch_text  = batch_text.to(device)
        batch_label = batch_label.to(device)
        optimizer.zero_grad()
        preds = model(batch_text)
        loss  = criterion(preds, batch_label)
        loss.backward()
        # Gradient clipping — stabilises RNN/LSTM training
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches  += 1
    return total_loss / n_batches if n_batches > 0 else 0.0


def evaluate(model, loader, criterion, device):
    """Evaluate model; returns (avg_loss, accuracy)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch_text, batch_label in loader:
            batch_text  = batch_text.to(device)
            batch_label = batch_label.to(device)
            preds       = model(batch_text)
            total_loss += criterion(preds, batch_label).item()
            correct    += ((preds > 0.5).float() == batch_label).sum().item()
            total      += batch_label.size(0)
    avg_loss = total_loss / len(loader) if len(loader) > 0 else 0.0
    accuracy = correct / total          if total > 0         else 0.0
    return avg_loss, accuracy


def train_model(model, epochs, optimizer, criterion, device):
    """Full training pipeline: train for `epochs` epochs; return test metrics."""
    # IMPORTANT: move model to device BEFORE creating optimizer
    model = model.to(device)
    for epoch in range(1, epochs + 1):
        train_loss          = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc   = evaluate(model, valid_loader, criterion, device)
        if epoch % max(1, epochs // 5) == 0 or epoch == epochs:
            print(f"  Epoch {epoch:3d}/{epochs} | train_loss={train_loss:.4f} "
                  f"| val_loss={val_loss:.4f} | val_acc={val_acc:.2%}")
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    return test_loss, test_acc


criterion = nn.BCELoss()
print("✓ Training utilities ready")


## 5. Warm-up: Baseline Vanilla RNN (SGD, 5 epochs)

**Objective:** Establish a lower-bound reference with a minimal RNN trained briefly.  
**Architecture:** Embedding → RNN → FC → Sigmoid  
**Optimizer:** SGD (lr = 0.001), 5 epochs


In [ ]:
print("=" * 70)
print("WARM-UP: BASELINE RNN  |  SGD  |  5 epochs")
print("=" * 70)

model_wu    = BaselineRNN(config.VOCAB_SIZE, config.EMBEDDING_DIM,
                          config.HIDDEN_DIM, config.DROPOUT).to(device)
opt_wu      = optim.SGD(model_wu.parameters(), lr=config.LEARNING_RATE)
loss_wu, acc_wu = train_model(model_wu, 5, opt_wu, criterion, device)

print(f"\nWarm-up — Test Loss: {loss_wu:.4f}  |  Test Accuracy: {acc_wu:.2%}")
print("=" * 70)


## 6. Experiment 1 — Optimizer Comparison

**Setup:** Baseline RNN, 5 epochs, lr = 0.001  
**Optimizers tested:** SGD · Adam · Adagrad


In [ ]:
print("=" * 70)
print("EXPERIMENT 1: OPTIMIZER COMPARISON  |  Baseline RNN  |  5 epochs")
print("=" * 70)

results_opt = []
for opt_name in ['SGD', 'Adam', 'Adagrad']:
    # Always move model to device BEFORE initialising optimizer
    model = BaselineRNN(config.VOCAB_SIZE, config.EMBEDDING_DIM,
                        config.HIDDEN_DIM, config.DROPOUT).to(device)
    if opt_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=config.LEARNING_RATE)
    elif opt_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
    else:
        optimizer = optim.Adagrad(model.parameters(), lr=config.LEARNING_RATE)

    print(f"\n--- {opt_name} ---")
    test_loss, test_acc = train_model(model, 5, optimizer, criterion, device)
    results_opt.append({'Optimizer': opt_name, 'Test Loss': test_loss, 'Test Accuracy': test_acc})
    print(f"Result  — Test Loss: {test_loss:.4f}  |  Test Accuracy: {test_acc:.2%}")

df_opt = pd.DataFrame(results_opt)
print("\n" + "=" * 70)
print(df_opt.to_string(index=False))


## 7. Experiment 2 — Effect of Training Epochs

**Setup:** Baseline RNN, Adam optimizer  
**Epochs tested:** 5 · 10 · 20 · 50


In [ ]:
print("=" * 70)
print("EXPERIMENT 2: EPOCH COMPARISON  |  Baseline RNN  |  Adam")
print("=" * 70)

results_epoch = []
for epochs in [5, 10, 20, 50]:
    model = BaselineRNN(config.VOCAB_SIZE, config.EMBEDDING_DIM,
                        config.HIDDEN_DIM, config.DROPOUT).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
    print(f"\n--- Epochs = {epochs} ---")
    test_loss, test_acc = train_model(model, epochs, optimizer, criterion, device)
    results_epoch.append({'Epochs': epochs, 'Test Loss': test_loss, 'Test Accuracy': test_acc})
    print(f"Result  — Test Loss: {test_loss:.4f}  |  Test Accuracy: {test_acc:.2%}")

df_epoch = pd.DataFrame(results_epoch)
print("\n" + "=" * 70)
print(df_epoch.to_string(index=False))


## 8. Experiment 3 — Architecture Comparison

**Setup:** Adam optimizer, 50 epochs, randomly initialised embeddings  
**Models:** FFN-1, FFN-2, FFN-3, CNN, LSTM, Bi-LSTM


In [ ]:
print("=" * 70)
print("EXPERIMENT 3: ARCHITECTURE COMPARISON  |  Adam  |  50 epochs")
print("=" * 70)

model_configs = [
    ('FFN-1',   FFN1   (config.VOCAB_SIZE, config.EMBEDDING_DIM, config.HIDDEN_DIM, config.DROPOUT)),
    ('FFN-2',   FFN2   (config.VOCAB_SIZE, config.EMBEDDING_DIM, config.HIDDEN_DIM, config.DROPOUT)),
    ('FFN-3',   FFN3   (config.VOCAB_SIZE, config.EMBEDDING_DIM, config.HIDDEN_DIM, config.DROPOUT)),
    ('CNN',     CNN    (config.VOCAB_SIZE, config.EMBEDDING_DIM, dropout=config.DROPOUT)),
    ('LSTM',    LSTMModel(config.VOCAB_SIZE, config.EMBEDDING_DIM, config.HIDDEN_DIM, config.DROPOUT)),
    ('Bi-LSTM', BiLSTM (config.VOCAB_SIZE, config.EMBEDDING_DIM, config.HIDDEN_DIM, config.DROPOUT)),
]

results_model = []
for model_name, model in model_configs:
    model     = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"\n--- {model_name}  ({num_params:,} parameters) ---")
    test_loss, test_acc = train_model(model, 50, optimizer, criterion, device)
    results_model.append({
        'Model': model_name, 'Parameters': num_params,
        'Test Loss': test_loss, 'Test Accuracy': test_acc
    })
    print(f"Result  — Test Loss: {test_loss:.4f}  |  Test Accuracy: {test_acc:.2%}")

df_model = pd.DataFrame(results_model)
print("\n" + "=" * 70)
print(df_model.to_string(index=False))


## 9. Results Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
fig.suptitle('RNN Sentiment Classification — Experimental Results',
             fontsize=15, fontweight='bold', y=1.02)

# ── Experiment 1: Optimizer Comparison ──────────────────────────────────────
colors_opt = ['#3498db', '#e74c3c', '#2ecc71']
bars1 = axes[0].bar(df_opt['Optimizer'], df_opt['Test Accuracy'] * 100,
                    color=colors_opt, edgecolor='black', linewidth=1.2, alpha=0.85)
axes[0].set_title('Exp 1: Optimizer Comparison\n(5 epochs, Baseline RNN)',
                  fontsize=12, fontweight='bold', pad=12)
axes[0].set_ylabel('Test Accuracy (%)', fontsize=11)
axes[0].set_ylim([0, 100])
axes[0].grid(axis='y', alpha=0.3, linestyle='--')
for bar in bars1:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width() / 2., h + 1.5,
                 f'{h:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)

# ── Experiment 2: Training Epochs ───────────────────────────────────────────
axes[1].plot(df_epoch['Epochs'], df_epoch['Test Accuracy'] * 100,
             marker='o', linewidth=2.5, markersize=10, color='#9b59b6')
axes[1].fill_between(df_epoch['Epochs'], df_epoch['Test Accuracy'] * 100, alpha=0.15, color='#9b59b6')
axes[1].set_title('Exp 2: Training Dynamics\n(Adam, Baseline RNN)',
                  fontsize=12, fontweight='bold', pad=12)
axes[1].set_xlabel('Number of Epochs', fontsize=11)
axes[1].set_ylabel('Test Accuracy (%)', fontsize=11)
axes[1].set_ylim([0, 100])
axes[1].grid(True, alpha=0.3, linestyle='--')
axes[1].set_xticks([5, 10, 20, 50])
for x, y in zip(df_epoch['Epochs'], df_epoch['Test Accuracy'] * 100):
    axes[1].text(x, y + 1.5, f'{y:.1f}%', ha='center', fontweight='bold', fontsize=9)

# ── Experiment 3: Architecture Comparison ───────────────────────────────────
colors_model = ['#95a5a6', '#95a5a6', '#95a5a6', '#f39c12', '#1abc9c', '#1abc9c']
bars3 = axes[2].bar(df_model['Model'], df_model['Test Accuracy'] * 100,
                    color=colors_model, edgecolor='black', linewidth=1.2, alpha=0.85)
axes[2].set_title('Exp 3: Architecture Comparison\n(Adam, 50 epochs)',
                  fontsize=12, fontweight='bold', pad=12)
axes[2].set_ylabel('Test Accuracy (%)', fontsize=11)
axes[2].set_ylim([0, 100])
axes[2].grid(axis='y', alpha=0.3, linestyle='--')
plt.sca(axes[2])
plt.xticks(rotation=25, ha='right')
for bar in bars3:
    h = bar.get_height()
    axes[2].text(bar.get_x() + bar.get_width() / 2., h + 1,
                 f'{h:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('experiment_results.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Figure saved as 'experiment_results.png'")


## 10. Summary Tables

In [ ]:
print("\n" + "=" * 70)
print("TABLE 1  —  Optimizer Comparison  (5 epochs, Baseline RNN)")
print("-" * 70)
tmp = df_opt.copy()
tmp['Test Accuracy'] = tmp['Test Accuracy'].map('{:.2%}'.format)
print(tmp.to_string(index=False))

print("\nTABLE 2  —  Epoch Comparison  (Adam, Baseline RNN)")
print("-" * 70)
tmp2 = df_epoch.copy()
tmp2['Test Accuracy'] = tmp2['Test Accuracy'].map('{:.2%}'.format)
print(tmp2.to_string(index=False))

print("\nTABLE 3  —  Architecture Comparison  (Adam, 50 epochs)")
print("-" * 70)
tmp3 = df_model.copy()
tmp3['Parameters']   = tmp3['Parameters'].map('{:,}'.format)
tmp3['Test Accuracy']= tmp3['Test Accuracy'].map('{:.2%}'.format)
print(tmp3.to_string(index=False))

# Export
df_opt.to_csv('results_optimizer.csv',     index=False)
df_epoch.to_csv('results_epochs.csv',      index=False)
df_model.to_csv('results_architectures.csv', index=False)
print("\n✓ Results exported to CSV files")


## 11. Key Findings & Conclusions

### Experiment 1 — Optimizers
All three optimisers converge to ~50% accuracy after only 5 epochs on a vanilla RNN,
confirming that the bottleneck is the architecture, not the optimiser.
Adam (50.65%) edges out SGD (50.36%) and Adagrad (50.42%), and is used for subsequent experiments.

### Experiment 2 — Training Epochs
Accuracy plateaus around 50–51% regardless of epoch count (5 → 50).
Test loss increases beyond 20 epochs, indicating overfitting.
Root cause: the vanishing gradient problem prevents the vanilla RNN from propagating
useful gradients over the 500-token sequences.

### Experiment 3 — Architectures
| Model   | Parameters | Test Accuracy | Params / 1% acc |
|---------|-----------|--------------|-----------------|
| FFN-1   | 17,664,513 | 81.06% | ~218k |
| FFN-2   | 17,697,281 | 81.40% | ~217k |
| FFN-3   | 17,705,473 | 81.75% | ~216k |
| CNN     | 1,357,401  | 86.28% | ~15.7k |
| **LSTM**| **1,675,521** | **88.17%** | ~19.0k |
| Bi-LSTM | 2,071,041  | 87.60% | ~23.6k |

- **LSTM** achieves the best accuracy (88.17%) with a moderate parameter count.
- **CNN** is the most parameter-efficient model (15.7k params per 1% accuracy).
- **FFNs** are the least efficient despite having >17M parameters — flattening discards sequential structure.
- **Bi-LSTM** underperforms LSTM under these settings (likely overfitting due to extra parameters).

### Overall Recommendation
For long-document sentiment classification, **LSTM** is the preferred choice when accuracy is the priority;
**CNN** is attractive when deployment efficiency matters.
A vanilla RNN is not competitive on this task, regardless of optimiser or training duration.
